# RF-DETR vs RT-DETR on DIVA-HisDB CB55

Fine-tune both detectors with Hugging Face `transformers.Trainer` on the existing CB55 COCO split and compare them on the untouched test set. Both models use exactly the same data, resizing/padding, optimizer, learning-rate schedule, epoch count, effective batch size, seed, checkpoint-selection metric, and COCO evaluation.

The final table reports **mAP@50** and **mAP@50–95**. In TorchMetrics/Hugging Face output, these are `map_50` and `map`, respectively. AP is computed with confidence threshold 0.0, as required for a full precision–recall curve.

Run the notebook top-to-bottom on a CUDA GPU. Model weights are downloaded from the Hugging Face Hub on first use.

In [1]:
# RF-DETR support is available in current Transformers releases. Restart the kernel
# after this cell if Jupyter reports that transformers was already imported.
%pip install -q -U "transformers>=5.14.1" "datasets>=4.0" "accelerate>=1.0" "torchmetrics>=1.4" pycocotools pillow pandas

Note: you may need to restart the kernel to use updated packages.


## Shared experiment configuration

The two checkpoints have different architectures and model-specific normalization statistics, but all tunable training and input geometry settings below are shared. No horizontal flip is used because mirrored manuscript text is not a plausible sample. Images retain their aspect ratio and are padded to a 640 × 640 canvas.

In [2]:
import gc
import json
import os
from collections import defaultdict
from dataclasses import dataclass
from functools import partial
from pathlib import Path

# Reduce allocator fragmentation across the two sequential full trainings below --
# without this, RT-DETR OOM'd partway through its run even though its own isolated
# peak (5.4GB) fit comfortably; residual fragmentation from RF-DETR's earlier run
# pushed it over. Must be set before the first CUDA allocation.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
import torchvision.transforms
from datasets import Dataset, DatasetDict, Image as HFImage
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from transformers import (
    AutoImageProcessor,
    AutoModelForObjectDetection,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.image_transforms import center_to_corners_format


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "00_data").is_dir() and (candidate / "50_modelling").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing 00_data and 50_modelling")


REPO_ROOT = find_repo_root(Path.cwd())
COCO_ROOT = REPO_ROOT / "00_data/DIVA-HisDB/coco_dataset_CB55"
IMAGE_ROOT = REPO_ROOT / "00_data/DIVA-HisDB/yolo_dataset_CB55/images"
OUTPUT_ROOT = REPO_ROOT / "80_models/02_2stage/diva-hisdb/detection/hf_detr_cb55"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_CHECKPOINTS = {
    "RF-DETR": "Roboflow/rf-detr-medium",
    "RT-DETR": "PekingU/rtdetr_r50vd",
}

# Recipe aligned with this repo's best-known CB55 detector: YOLOv8+ResNet34-P2
# @ imgsz=1568 (80_models/.../_exploratory_imgsz_sweep/resnet34_cb55_1568_023,
# test mAP50=0.978). imgsz is dialed back to 1408 (still divisible by both
# checkpoints' 32px stride) after 1568 OOM'd RT-DETR mid-run despite fitting
# in isolation -- 1408's isolated bf16 peak (4.3GB RT-DETR / 3.1GB RF-DETR)
# leaves more headroom for two full sequential trainings in one process. That
# run's HSV color-jitter augmentation (hsv_h/s/v) is carried over below; its
# horizontal-flip default (fliplr=0.5) is deliberately NOT carried over --
# mirrored manuscript text is not a plausible sample, as already noted in
# this notebook. Epochs/lr/effective-batch/clip stay at the values that got
# RF-DETR to 0.968/0.963 val/test mAP50 via the native rfdetr package earlier
# this session.
IMAGE_SIZE = 1408
NUM_EPOCHS = 100
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.0
MAX_GRAD_NORM = 0.1
SEED = 42
NUM_WORKERS = 2

# Color-jitter augmentation approximating YOLO's hsv_h=0.015/hsv_s=0.7/hsv_v=0.4
# (image-only, so it's safe to apply without touching box coordinates).
COLOR_JITTER_BRIGHTNESS = 0.4
COLOR_JITTER_CONTRAST = 0.4
COLOR_JITTER_SATURATION = 0.7
COLOR_JITTER_HUE = 0.015

BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
FP16 = bool(torch.cuda.is_available() and not BF16)
set_seed(SEED)

print(f"Repository: {REPO_ROOT}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}; bf16={BF16}; fp16={FP16}")
print(f"Effective batch size: {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
if not torch.cuda.is_available():
    print("WARNING: training these detectors on CPU is impractically slow.")

Repository: /home/artur/Thesis/fsl-tl-manuscripts
Device: cuda; bf16=True; fp16=False
Effective batch size: 16


## Load the local COCO annotations

CB55 category id 1 is remapped to the contiguous model label 0 (`TextLine`). Bounding boxes are clipped to the image extent and degenerate boxes are rejected. The existing train, validation, and test split is preserved.

In [3]:
def load_coco_split(split: str, category_id_to_label: dict[int, int]) -> Dataset:
    annotation_path = COCO_ROOT / f"{split}.json"
    payload = json.loads(annotation_path.read_text(encoding="utf-8"))

    annotations_by_image = defaultdict(list)
    for annotation in payload["annotations"]:
        annotations_by_image[annotation["image_id"]].append(annotation)

    records = []
    for image_info in sorted(payload["images"], key=lambda item: item["id"]):
        width, height = int(image_info["width"]), int(image_info["height"])
        image_path = IMAGE_ROOT / split / image_info["file_name"]
        if not image_path.is_file():
            raise FileNotFoundError(image_path)

        boxes, labels, areas, annotation_ids = [], [], [], []
        for annotation in annotations_by_image[image_info["id"]]:
            x, y, box_width, box_height = map(float, annotation["bbox"])
            x = min(max(x, 0.0), float(width))
            y = min(max(y, 0.0), float(height))
            box_width = min(box_width, width - x)
            box_height = min(box_height, height - y)
            if box_width <= 0 or box_height <= 0:
                continue
            boxes.append([x, y, box_width, box_height])
            labels.append(category_id_to_label[int(annotation["category_id"])])
            areas.append(box_width * box_height)
            annotation_ids.append(int(annotation["id"]))

        if not boxes:
            raise ValueError(f"{image_path} has no valid boxes")
        records.append({
            "image_id": int(image_info["id"]),
            "image": str(image_path),
            "width": width,
            "height": height,
            "objects": {
                "id": annotation_ids,
                "bbox": boxes,
                "category": labels,
                "area": areas,
            },
        })

    return Dataset.from_list(records).cast_column("image", HFImage())


category_payload = json.loads((COCO_ROOT / "train.json").read_text(encoding="utf-8"))
categories = sorted(category_payload["categories"], key=lambda item: item["id"])
category_id_to_label = {int(category["id"]): index for index, category in enumerate(categories)}
id2label = {index: category["name"] for index, category in enumerate(categories)}
label2id = {name: index for index, name in id2label.items()}

raw_datasets = DatasetDict({
    split: load_coco_split(split, category_id_to_label)
    for split in ("train", "val", "test")
})

split_summary = []
for split, dataset in raw_datasets.items():
    number_of_boxes = sum(len(objects["bbox"]) for objects in dataset["objects"])
    split_summary.append({"split": split, "pages": len(dataset), "boxes": number_of_boxes})
pd.DataFrame(split_summary).set_index("split")

,pages,boxes
split,,
train,20,1507
val,10,901
test,10,845


## Model-specific processor, shared geometry

Each checkpoint keeps its published normalization statistics. Both processors use the same longest-side resize and square padding. The annotations passed to the processor remain COCO `(x, y, width, height)` boxes.

In [4]:
def format_annotations_as_coco(image_id, categories, areas, boxes):
    return {
        "image_id": image_id,
        "annotations": [
            {
                "image_id": image_id,
                "category_id": category,
                "iscrowd": 0,
                "area": area,
                "bbox": list(box),
            }
            for category, area, box in zip(categories, areas, boxes)
        ],
    }


TRAIN_COLOR_JITTER = torchvision.transforms.ColorJitter(
    brightness=COLOR_JITTER_BRIGHTNESS,
    contrast=COLOR_JITTER_CONTRAST,
    saturation=COLOR_JITTER_SATURATION,
    hue=COLOR_JITTER_HUE,
)


def transform_batch(examples, image_processor, augment: bool):
    images, annotations = [], []
    for image_id, image, objects in zip(examples["image_id"], examples["image"], examples["objects"]):
        image = image.convert("RGB")
        if augment:
            image = TRAIN_COLOR_JITTER(image)
        images.append(np.array(image, copy=True))
        annotations.append(format_annotations_as_coco(
            image_id, objects["category"], objects["area"], objects["bbox"]
        ))

    # Keep pixel_mask: aspect-ratio padding is excluded from both models' attention.
    return image_processor(images=images, annotations=annotations, return_tensors="pt")


def make_image_processor(checkpoint: str):
    return AutoImageProcessor.from_pretrained(
        checkpoint,
        do_resize=True,
        size={"max_height": IMAGE_SIZE, "max_width": IMAGE_SIZE},
        do_pad=True,
        pad_size={"height": IMAGE_SIZE, "width": IMAGE_SIZE},
    )


def make_processed_datasets(image_processor) -> DatasetDict:
    return DatasetDict({
        split: dataset.with_transform(
            partial(transform_batch, image_processor=image_processor, augment=(split == "train"))
        )
        for split, dataset in raw_datasets.items()
    })


def collate_fn(batch):
    result = {
        "pixel_values": torch.stack([item["pixel_values"] for item in batch]),
        "labels": [item["labels"] for item in batch],
    }
    if "pixel_mask" in batch[0]:
        result["pixel_mask"] = torch.stack([item["pixel_mask"] for item in batch])
    return result

## COCO mAP

Targets and predictions are converted to absolute `xyxy` coordinates before evaluation. `MeanAveragePrecision` uses the standard COCO IoU thresholds 0.50:0.05:0.95 and a maximum of 100 detections per image.

In [5]:
@dataclass
class DetectionOutput:
    logits: torch.Tensor
    pred_boxes: torch.Tensor


def original_size(target):
    values = np.atleast_1d(np.asarray(target["orig_size"])).flatten()
    if len(values) < 2:
        raise ValueError(f"Invalid orig_size: {target['orig_size']}")
    return int(values[0]), int(values[1])


def normalized_cxcywh_to_absolute_xyxy(boxes, image_size):
    height, width = image_size
    boxes = center_to_corners_format(boxes)
    return boxes * torch.tensor([[width, height, width, height]], dtype=boxes.dtype)


@torch.no_grad()
def compute_metrics(evaluation_results, image_processor):
    predictions, targets = evaluation_results.predictions, evaluation_results.label_ids
    image_sizes = []
    processed_targets = []
    processed_predictions = []

    for target_batch in targets:
        batch_sizes = []
        for target in target_batch:
            size = original_size(target)
            batch_sizes.append(size)
            boxes = normalized_cxcywh_to_absolute_xyxy(torch.as_tensor(target["boxes"]), size)
            labels = torch.as_tensor(target["class_labels"], dtype=torch.int64)
            processed_targets.append({"boxes": boxes, "labels": labels})
        image_sizes.append(torch.tensor(batch_sizes))

    # With eval_do_concat_batches=False, predictions contains one tuple per batch.
    # AutoModelForObjectDetection exposes loss_dict, logits, and pred_boxes in these positions.
    for prediction_batch, target_sizes in zip(predictions, image_sizes):
        batch_logits, batch_boxes = prediction_batch[1], prediction_batch[2]
        outputs = DetectionOutput(
            logits=torch.as_tensor(batch_logits),
            pred_boxes=torch.as_tensor(batch_boxes),
        )
        processed_predictions.extend(image_processor.post_process_object_detection(
            outputs, threshold=0.0, target_sizes=target_sizes
        ))

    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=False)
    metric.update(processed_predictions, processed_targets)
    return {key: round(value.item(), 6) for key, value in metric.compute().items()}


## Train and evaluate both detectors

Validation `mAP@50–95` selects the best checkpoint. After training, that checkpoint is evaluated once on `test`; both the full metric JSON and the model/processor are saved below `80_models/02_2stage/diva-hisdb/detection/hf_detr_cb55`.

In [6]:
from transformers.models.rt_detr.modeling_rt_detr import replace_batch_norm as _rtdetr_replace_batch_norm


def freeze_stray_batch_norms(model: torch.nn.Module) -> int:
    """Freeze any live nn.BatchNorm2d left in the model (e.g. RT-DETR's hybrid-encoder
    conv-BN blocks in encoder_input_proj/encoder/decoder_input_proj -- its ResNet
    backbone is already frozen to RTDetrFrozenBatchNorm2d by HF's own
    freeze_backbone_batch_norms default, but those neck/encoder BatchNorms are not).

    At PER_DEVICE_BATCH_SIZE=1 (needed to fit these models at IMAGE_SIZE on this GPU),
    gradient accumulation only averages gradients over several size-1 forward passes --
    it does NOT give BatchNorm a real batch to compute statistics from. Each live BN
    layer keeps recomputing mean/var from a single sample every step, which is pure
    noise and progressively corrupts the encoder's normalization over 100 epochs.

    Uses RT-DETR's own replace_batch_norm (recursively swaps nn.BatchNorm2d for the
    non-trainable RTDetrFrozenBatchNorm2d class, copying the pretrained running
    stats/affine params) rather than just calling .eval() -- Trainer.train() calls
    model.train() every epoch, which would silently flip a merely-.eval()'d BN back
    to training mode; swapping the module class entirely is immune to that. No-op for
    RF-DETR (LayerNorm-only ViT backbone, no BatchNorm2d anywhere).
    """
    before = sum(1 for m in model.modules() if isinstance(m, torch.nn.BatchNorm2d))
    _rtdetr_replace_batch_norm(model)
    after = sum(1 for m in model.modules() if isinstance(m, torch.nn.BatchNorm2d))
    return before - after


def train_and_evaluate(model_name: str, checkpoint: str):
    # Reset all Trainer/DataLoader RNGs so both runs receive the same stochastic ordering.
    set_seed(SEED)
    run_name = model_name.lower().replace("-", "_")
    output_dir = OUTPUT_ROOT / run_name

    image_processor = make_image_processor(checkpoint)
    datasets = make_processed_datasets(image_processor)
    model = AutoModelForObjectDetection.from_pretrained(
        checkpoint,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )
    n_frozen_bn = freeze_stray_batch_norms(model)
    print(f"{model_name}: froze {n_frozen_bn} live BatchNorm2d layer(s) (batch_size=1 fine-tuning)")

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        run_name=f"cb55-{run_name}",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        optim="adamw_torch",
        max_grad_norm=MAX_GRAD_NORM,
        bf16=BF16,
        fp16=FP16,
        dataloader_num_workers=NUM_WORKERS,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        metric_for_best_model="eval_map",
        greater_is_better=True,
        load_best_model_at_end=True,
        save_total_limit=2,
        remove_unused_columns=False,
        eval_do_concat_batches=False,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=datasets["train"],
        eval_dataset=datasets["val"],
        processing_class=image_processor,
        data_collator=collate_fn,
        compute_metrics=partial(compute_metrics, image_processor=image_processor),
    )

    train_result = trainer.train()
    trainer.save_model(str(output_dir / "best_model"))
    image_processor.save_pretrained(str(output_dir / "best_model"))
    validation_metrics = trainer.evaluate(datasets["val"], metric_key_prefix="validation")
    test_metrics = trainer.evaluate(datasets["test"], metric_key_prefix="test")

    record = {
        "model": model_name,
        "checkpoint": checkpoint,
        "validation_mAP@50": validation_metrics["validation_map_50"],
        "validation_mAP@50-95": validation_metrics["validation_map"],
        "test_mAP@50": test_metrics["test_map_50"],
        "test_mAP@50-95": test_metrics["test_map"],
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "train_runtime_seconds": train_result.metrics.get("train_runtime"),
    }
    (output_dir / "metrics_summary.json").write_text(
        json.dumps({"summary": record, "validation": validation_metrics, "test": test_metrics}, indent=2),
        encoding="utf-8",
    )

    del trainer, model, datasets, image_processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return record


In [7]:
results = []
for model_name, checkpoint in MODEL_CHECKPOINTS.items():
    print(f"\n{'=' * 80}\nTraining {model_name} from {checkpoint}\n{'=' * 80}")
    results.append(train_and_evaluate(model_name, checkpoint))


Training RF-DETR from Roboflow/rf-detr-medium


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

[transformers] RfDetrForObjectDetection LOAD REPORT from: Roboflow/rf-detr-medium
Key                                       | Status   |                                                                                        
------------------------------------------+----------+----------------------------------------------------------------------------------------
class_embed.bias                          | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([91]) vs model:torch.Size([1])          
model.enc_out_class_embed.{0...12}.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([91]) vs model:torch.Size([1])          
model.enc_out_class_embed.{0...12}.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([91, 256]) vs model:torch.Size([1, 256])
class_embed.weight                        | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([91, 256]) vs model:torch.Size([1, 256])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did n

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


RF-DETR: froze 0 live BatchNorm2d layer(s) (batch_size=1 fine-tuning)


...

[transformers] There were unexpected keys in the checkpoint model loaded: ['model.encoder.encoder.0.layers.0.fc1.bias', 'model.encoder.encoder.0.layers.0.fc1.weight', 'model.encoder.encoder.0.layers.0.fc2.bias', 'model.encoder.encoder.0.layers.0.fc2.weight', 'model.encoder.encoder.0.layers.0.final_layer_norm.bias', 'model.encoder.encoder.0.layers.0.final_layer_norm.weight', 'model.encoder.encoder.0.layers.0.self_attn.k_proj.bias', 'model.encoder.encoder.0.layers.0.self_attn.k_proj.weight', 'model.encoder.encoder.0.layers.0.self_attn.out_proj.bias', 'model.encoder.encoder.0.layers.0.self_attn.out_proj.weight', 'model.encoder.encoder.0.layers.0.self_attn.q_proj.bias', 'model.encoder.encoder.0.layers.0.self_attn.q_proj.weight', 'model.encoder.encoder.0.layers.0.self_attn.v_proj.bias', 'model.encoder.encoder.0.layers.0.self_attn.v_proj.weight', 'model.encoder.encoder.0.layers.0.self_attn_layer_norm.bias', 'model.encoder.encoder.0.layers.0.self_attn_layer_norm.weight', 'model.decoder.layers

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/artur/Thesis/fsl-tl-manuscripts/.venv/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


Training Loss,Validation Loss,Epoch,Map,Map 50,Map 75,Map Small,Map Medium,Map Large,Mar 1,Mar 10,Mar 100,Mar Small,Mar Medium,Mar Large,Map Per Class,Mar 100 Per Class,Classes
7.579099,6.093131,100,0.739863,0.853584,0.840546,-1.000000,0.050495,0.741104,0.010100,0.100999,0.801221,-1.000000,0.300000,0.802895,-1.000000,-1.000000,0


Training Loss,Validation Loss,Epoch,Map,Map 50,Map 75,Map Small,Map Medium,Map Large,Mar 1,Mar 10,Mar 100,Mar Small,Mar Medium,Mar Large,Map Per Class,Mar 100 Per Class,Classes
7.579099,6.271127,100,0.713167,0.832032,0.813128,-1.000000,0.201980,0.713681,0.010533,0.106391,0.779290,-1.000000,0.400000,0.781094,-1.000000,-1.000000,0


## Final CB55 test report

These are held-out **test** results from the validation-selected checkpoint. Higher is better. The CSV is saved beside the two model directories.

In [8]:
report = (
    pd.DataFrame(results)
    .set_index("model")[["test_mAP@50", "test_mAP@50-95", "validation_mAP@50", "validation_mAP@50-95"]]
    .sort_values("test_mAP@50-95", ascending=False)
)
report.to_csv(OUTPUT_ROOT / "rf_detr_vs_rt_detr_cb55_metrics.csv")
display(report.style.format("{:.4f}").highlight_max(subset=["test_mAP@50", "test_mAP@50-95"], axis=0))

,test_mAP@50,test_mAP@50-95,validation_mAP@50,validation_mAP@50-95
model,,,,
RT-DETR,0.8320,0.7132,0.8536,0.7399
RF-DETR,0.4964,0.2094,0.4297,0.1843


### Reproducibility note

The shared configuration controls the comparison, but exact floating-point results can still vary slightly across GPU models, CUDA/cuDNN versions, and Transformers/PyTorch builds. The test split is never used for checkpoint selection.